# Chapitre 6 — Préentraînement, transfert et efficacité
## Atelier corrigé · M2 IA
PyTorch CPU, NumPy, Matplotlib. Prédire → mesurer → expliquer.


In [ ]:
import math, time, copy
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(4)
print('PyTorch :', torch.__version__)


## Atelier 1 · Un langage synthétique à quatre thèmes
Le thème n’est jamais donné au modèle de langage.


In [ ]:
# CELLULE 2 — Un « langage » synthétique : quatre thèmes, chacun avec ses transitions.
V, T, K = 24, 32, 4                       # vocabulaire, longueur, nombre de thèmes
g = torch.Generator().manual_seed(0)
base = 2.0 * torch.randn(V, V, generator=g)           # structure commune à tous les thèmes
TRANS = (base + 1.5 * torch.randn(K, V, V, generator=g)).softmax(-1)   # (K, V, V) : un écart propre à chaque thème

def sample(n, seed):
    gen = torch.Generator().manual_seed(seed)
    z = torch.randint(0, K, (n,), generator=gen)
    x = torch.empty(n, T, dtype=torch.long)
    x[:, 0] = torch.randint(0, V, (n,), generator=gen)
    for t in range(1, T):
        x[:, t] = torch.multinomial(TRANS[z, x[:, t - 1]], 1, generator=gen).squeeze(1)
    return x, z

X_pre, _ = sample(20000, seed=1)          # préentraînement : aucune étiquette utilisée
X_lm_val, _ = sample(2000, seed=2)        # mesure de la perte de langage (oubli)
X_tr, z_tr = sample(64, seed=3)           # tâche cible : 64 exemples étiquetés
X_te, z_te = sample(4000, seed=4)
print('Préentraînement', tuple(X_pre.shape), '; cible', tuple(X_tr.shape), '; test', tuple(X_te.shape))


### Préentraîner un petit GPT
Prédisez la perte de validation : plus proche de 1,860 ou de 1,520 ?


In [ ]:
# CELLULE 3 — Un petit décodeur causal (type GPT) et son préentraînement.
class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h = h
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.fc1, self.fc2 = nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
    def forward(self, x):
        B, n, d = x.shape
        q, k, v = self.qkv(self.ln1(x)).view(B, n, 3, self.h, d // self.h).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.proj(a.transpose(1, 2).reshape(B, n, d))
        return x + self.fc2(F.gelu(self.fc1(self.ln2(x))))

class TinyGPT(nn.Module):
    def __init__(self, d=64, h=4, layers=2):
        super().__init__()
        self.emb, self.pos = nn.Embedding(V, d), nn.Parameter(0.02 * torch.randn(T, d))
        self.blocks = nn.ModuleList(Block(d, h) for _ in range(layers))
        self.ln, self.lm_head = nn.LayerNorm(d), nn.Linear(d, V)
    def features(self, x):
        h = self.emb(x) + self.pos[:x.shape[1]]
        for b in self.blocks:
            h = b(h)
        return self.ln(h)                          # (B, n, d)
    def forward(self, x):
        return self.lm_head(self.features(x))

def lm_loss(model, X):
    with torch.no_grad():
        logits = model(X[:, :-1])
        return F.cross_entropy(logits.reshape(-1, V), X[:, 1:].reshape(-1)).item()

def pretrain(steps=2500, seed=0):
    torch.manual_seed(seed)
    model = TinyGPT()
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=steps, pct_start=.1)
    gen, history = torch.Generator().manual_seed(seed), []
    for step in range(steps):
        xb = X_pre[torch.randint(0, len(X_pre), (64,), generator=gen)]
        loss = F.cross_entropy(model(xb[:, :-1]).reshape(-1, V), xb[:, 1:].reshape(-1))
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if step % 50 == 0:
            history.append((step, loss.item()))
    return model, history

t0 = time.time()
pretrained, pre_history = pretrain()
print(f'Préentraînement : {time.time()-t0:.0f} s ; perte LM validation = {lm_loss(pretrained, X_lm_val):.3f}',
      f'(uniforme : log V = {math.log(V):.3f})')


## Atelier 2 · LoRA et tête de classification
Vérifiez qu’avec B = 0 le modèle adapté calcule exactement la même chose que le modèle préentraîné.


In [ ]:
# CELLULE 4 — LoRA : W = W₀ + (α/r)·B A, avec B initialisée à zéro.
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha=None):
        super().__init__()
        self.base, self.r, self.scale = base, r, (alpha or 2 * r) / r
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A.T) @ self.B.T
    def merged_weight(self):
        return self.base.weight + self.scale * self.B @ self.A

def add_lora(model, r):
    for blk in model.blocks:
        for name in ('qkv', 'proj', 'fc1', 'fc2'):
            setattr(blk, name, LoRALinear(getattr(blk, name), r))
    return model

class Classifier(nn.Module):
    """Tronc (préentraîné ou non) + tête linéaire sur la moyenne des positions."""
    def __init__(self, trunk):
        super().__init__()
        self.trunk, self.head = trunk, nn.Linear(64, K)
    def forward(self, x):
        return self.head(self.trunk.features(x).mean(1))

def trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


### Cinq stratégies, 64 exemples, trois graines
Classez les stratégies avant d’exécuter.


In [ ]:
# CELLULE 5 — Cinq stratégies, trois graines, même budget : 300 mises à jour sur 64 exemples.
def adapt(method, seed, Xs=None, zs=None, steps=300):
    Xs, zs = (X_tr, z_tr) if Xs is None else (Xs, zs)
    torch.manual_seed(seed)
    trunk = TinyGPT() if method == 'scratch' else copy.deepcopy(pretrained)
    if method == 'probe' or method.startswith('lora'):
        for p in trunk.parameters():
            p.requires_grad_(False)                 # tronc gelé
    if method.startswith('lora'):
        add_lora(trunk, int(method.split('-')[1]))
    model = Classifier(trunk)
    lr = {'scratch': 1e-3, 'probe': 1e-2, 'full': 3e-4}.get(method, 3e-3)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=.01)
    gen = torch.Generator().manual_seed(seed)
    for step in range(steps):
        idx = torch.randperm(len(Xs), generator=gen)[:64]
        loss = F.cross_entropy(model(Xs[idx]), zs[idx])
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    with torch.no_grad():
        acc = (model(X_te).argmax(-1) == z_te).float().mean().item()
    return model, acc

methods = ['scratch', 'probe', 'full', 'lora-1', 'lora-4', 'lora-16']
results = {}
for m in methods:
    runs = [adapt(m, s) for s in range(3)]
    results[m] = dict(acc=[a for _, a in runs], params=trainable(runs[0][0]), model=runs[0][0])
    print(f'{m:8s} accuracy {np.mean(results[m]["acc"]):.3f} ± {np.std(results[m]["acc"]):.3f}'
          f'   paramètres entraînés : {results[m]["params"]:6d}')


### Oubli
Que devient le modèle de langage après chaque adaptation ?


In [ ]:
# CELLULE 6 — Oubli : que devient le modèle de langage après adaptation ?
def lm_loss_of(model):
    trunk = model.trunk
    with torch.no_grad():
        logits = trunk.lm_head(trunk.features(X_lm_val[:, :-1]))   # même tête LM, tronc adapté
        return F.cross_entropy(logits.reshape(-1, V), X_lm_val[:, 1:].reshape(-1)).item()

forgetting = {'préentraîné': lm_loss(pretrained, X_lm_val)}
for m in ['probe', 'full', 'lora-4']:
    forgetting[m] = lm_loss_of(results[m]['model'])
lora_layers = [mod for mod in results['lora-4']['model'].modules() if isinstance(mod, LoRALinear)]
scales = [mod.scale for mod in lora_layers]
for mod in lora_layers:
    mod.scale = 0.                                   # on retire B·A
forgetting['lora-4, adaptateurs retirés'] = lm_loss_of(results['lora-4']['model'])
for mod, s in zip(lora_layers, scales):
    mod.scale = s
for k, v in forgetting.items():
    print(f'{k:28s} perte LM : {v:.3f}')


## Atelier 3 · Efficacité en étiquettes (≈ 3 min)


In [ ]:
# CELLULE 7 — Efficacité en étiquettes : la valeur du préentraînement dépend de n.
X_big, z_big = sample(1024, seed=5)
label_curve = {m: [] for m in ['scratch', 'probe', 'full', 'lora-4']}
for n in [16, 64, 256, 1024]:
    for m in label_curve:
        label_curve[m].append((n, adapt(m, 0, X_big[:n], z_big[:n])[1]))
    print(n, {m: round(v[-1][1], 3) for m, v in label_curve.items()})
for m, pts in label_curve.items():
    plt.semilogx(*zip(*pts), 'o-', label=m)
plt.xlabel('Exemples étiquetés'); plt.ylabel('Accuracy test'); plt.legend(); plt.show()


### Mémoire : ce qu’Adam stocke vraiment


In [ ]:
# CELLULE 8 — Mémoire : paramètres entraînés et états d'Adam, mesurés plutôt que supposés.
def optimizer_state_numel(model):
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
    loss = F.cross_entropy(model(X_tr[:8]), z_tr[:8]); loss.backward(); opt.step()
    return sum(v.numel() for s in opt.state.values() for v in s.values() if torch.is_tensor(v) and v.dim() > 0)

for m in ['full', 'lora-4']:
    model = adapt(m, 0, steps=1)[0]
    print(f'{m:7s} entraînés : {trainable(model):7d} ; états Adam : {optimizer_state_numel(model):7d}')

def training_bytes(n_params, n_trainable, weight_bytes=2):
    """Poids (précision de stockage) + gradients, états Adam (m, v) et copie fp32 des paramètres entraînés."""
    return n_params * weight_bytes + n_trainable * (2 + 4 + 4 + 4)
N = 7e9
for label, wb, frac in [('ajustement complet, 16 bits', 2, 1.), ('LoRA, base 16 bits', 2, .005), ('QLoRA, base 4 bits', .5, .005)]:
    print(f'{label:30s} ≈ {training_bytes(N, frac * N, wb) / 2**30:6.1f} Gio (hors activations)')


### Quantification après entraînement


In [ ]:
# CELLULE 9 — Quantification après entraînement : arrondir les poids et mesurer.
def fake_quant(W, bits, group=None):
    """Quantification symétrique absmax, par tenseur (group=None) ou par groupes de colonnes."""
    qmax = 2 ** (bits - 1) - 1
    Wg = W.reshape(-1, group) if group else W.reshape(1, -1)
    scale = Wg.abs().amax(dim=1, keepdim=True).clamp_min(1e-12) / qmax
    return (torch.round(Wg / scale).clamp(-qmax, qmax) * scale).reshape(W.shape)

def quantized(model, bits, group=None):
    q = copy.deepcopy(model)
    with torch.no_grad():
        for blk in q.blocks:
            for lin in (blk.qkv, blk.proj, blk.fc1, blk.fc2):
                lin.weight.copy_(fake_quant(lin.weight, bits, group))
    return q

print(f'{"référence 32 bits":24s} perte LM {lm_loss(pretrained, X_lm_val):.4f}')
for bits, group in [(8, None), (4, None), (4, 32), (3, None), (3, 32)]:
    name = f'int{bits} ' + ('par tenseur' if group is None else f'groupes de {group}')
    print(f'{name:24s} perte LM {lm_loss(quantized(pretrained, bits, group), X_lm_val):.4f}')


### Une valeur aberrante


In [ ]:
# CELLULE 10 — Une valeur aberrante suffit à ruiner la quantification par tenseur.
r = np.random.default_rng(0)
W = r.normal(0, .02, size=(256, 256))
W_out = W.copy(); W_out[3, 7] = 2.0                       # une seule valeur 100 fois plus grande
def rel_err(W, bits, group=None):
    Wq = fake_quant(torch.tensor(W), bits, group).numpy()
    return np.linalg.norm(W - Wq) / np.linalg.norm(W)
for name, M in [('sans valeur aberrante', W), ('avec une valeur aberrante', W_out)]:
    print(f'{name:26s} int8/tenseur {rel_err(M, 8):.4f} | int4/tenseur {rel_err(M, 4):.4f} | int4/groupes 32 {rel_err(M, 4, 32):.4f}')


## Atelier 4 · Distillation (≈ 1 min)
Refaites l’expérience sur plusieurs graines avant de conclure.


In [ ]:
# CELLULE 11 — Distillation : un élève deux fois plus étroit apprend des probabilités du maître.
def distill(alpha, T_kd=2.0, steps=1500, seed=0):
    torch.manual_seed(seed)
    student = TinyGPT(d=32, h=2, layers=1)
    opt = torch.optim.AdamW(student.parameters(), lr=3e-3, weight_decay=.01)
    gen = torch.Generator().manual_seed(seed)
    for step in range(steps):
        xb = X_pre[torch.randint(0, len(X_pre), (64,), generator=gen)]
        s = student(xb[:, :-1]).reshape(-1, V)
        hard = F.cross_entropy(s, xb[:, 1:].reshape(-1))
        with torch.no_grad():
            t = pretrained(xb[:, :-1]).reshape(-1, V)
        soft = F.kl_div(F.log_softmax(s / T_kd, -1), F.softmax(t / T_kd, -1), reduction='batchmean') * T_kd**2
        loss = (1 - alpha) * hard + alpha * soft
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return student

print('Paramètres maître / élève :', sum(p.numel() for p in pretrained.parameters()),
      sum(p.numel() for p in TinyGPT(d=32, h=2, layers=1).parameters()))
for alpha in [0., .5, .9]:
    print(f'α = {alpha:.1f} : perte LM de l’élève {lm_loss(distill(alpha), X_lm_val):.3f}')
